# Мацак Елизавета Сергеевна

# 1

In [1]:
import csv
from datetime import datetime

transactions: list[dict] = []
with open("transactions.csv", encoding="utf-8", newline="") as file:
    file_dict = csv.DictReader(file)
    for row in file_dict:
        transactions.append({
            "transaction_id": row["transaction_id"],
            "client_id": row["client_id"],
            "transaction_sum_roubles": row["transaction_sum_roubles"],
            "purpose": row["purpose"],
            "card_type": row["card_type"],
            "transaction_datetime": row["transaction_datetime"]
        })

In [2]:
from collections import defaultdict, Counter

total_sum: float = sum(map(lambda tx: float(tx["transaction_sum_roubles"]), transactions))

card_totals: dict[str, float] = defaultdict(float)
purpose_totals: dict[str, float] = defaultdict(float)
purpose_counts: Counter[str] = Counter()

for tx in transactions:
    amount = float(tx["transaction_sum_roubles"])
    card_totals[tx["card_type"]] += amount
    purpose_totals[tx["purpose"]] += amount
    purpose_counts[tx["purpose"]] += 1

card_totals = dict(card_totals)
purpose_averages: dict[str, float] = {purpose: amount / purpose_counts[purpose] for purpose, amount in purpose_totals.items()}

In [3]:
print(len(transactions))  # 30000
print(f"Общая сумма: {total_sum:.0f}")

print('\nCумма в разрезе кредитных и дебетовых карт:')
for card_type, amount in card_totals.items():
    print(f"{card_type}: {amount:.0f}")

print('\nCумма в разрезе целей:')
for purpose, average in purpose_averages.items():
    print(f"{purpose}: {average:.0f}")

30000
Общая сумма: 7457125766

Cумма в разрезе кредитных и дебетовых карт:
debit: 3715193149
credit: 3741932618

Cумма в разрезе целей:
Utilities: 245235
Electronics: 247538
Subscription: 250184
Clothing: 251430
Transfer: 249343
Fuel: 245407
Entertainment: 249597
Groceries: 249481
Restaurant: 251183
Online purchase: 246441


# 2

In [4]:
import time
from collections import OrderedDict


def _slow_square(number: int) -> int:
    time.sleep(3)
    return number * number

square_cache: OrderedDict[int, int] = OrderedDict()

def cached_square(number: int) -> int:
    if number not in square_cache:
        square_cache[number] = _slow_square(number)
    return square_cache[number]

In [5]:
for number in [2, 2, 3, 2, 5]:
    start = time.perf_counter()
    result = cached_square(number)
    elapsed = time.perf_counter() - start
    print(f"{number}: {result}, время: {elapsed:.2f}")

2: 4, время: 3.01
2: 4, время: 0.00
3: 9, время: 3.00
2: 4, время: 0.00
5: 25, время: 3.00


# 3

In [6]:
def polynomial_formatter(*args: int | float) -> str:
    parts: list[str] = []
    for i, coefficient in enumerate(args):
        if coefficient == 0:
            continue
        power = len(args) - i - 1
        value = abs(coefficient)
        if power == 0:
            term = str(value)
        else:
            variable = "x" if power == 1 else f"x^{power}"
            term = variable if value == 1 else f"{value}*{variable}"
        if not parts:
            term = f"-{term}" if coefficient < 0 else term
        else:
            term = f"- {term}" if coefficient < 0 else f"+ {term}"
        parts.append(term)
    expression = " ".join(parts) if parts else "0"
    return f"{expression} = 0"

In [7]:
print(polynomial_formatter(3, 2, 1))   
# 3*x^2 + 2*x + 1 = 0
print(polynomial_formatter(1, -2, 1))  # x^2 - 2*x + 1 = 0
print(polynomial_formatter(0, 2, 1))   # 2*x + 1 = 0
print(polynomial_formatter(-1, 0, 1))  # -x^2 + 1 = 0
print(polynomial_formatter(1, 0))     # x = 0
print(polynomial_formatter(0, 0, 0))  # 0 = 0
print(polynomial_formatter())         # 0 = 0

3*x^2 + 2*x + 1 = 0
x^2 - 2*x + 1 = 0
2*x + 1 = 0
-x^2 + 1 = 0
x = 0
0 = 0
0 = 0


# 4

Первые 20 совпадений в порядке файла, как в примере из условия.

In [8]:
def select_top_20(data: list[dict] | None = None, **filters) -> list[dict]:
    if data is None:
        data = transactions
    result: list[dict] = []
    for tx in data:
        if all(key in tx and tx[key] == value for key, value in filters.items()):
            result.append(tx)
            if len(result) == 20:
                break
    return result

In [9]:
print(select_top_20(transactions, transaction_id=9))
print(select_top_20(transaction_id=9))

fuel_transactions = select_top_20(purpose="Fuel", card_type="debit")
print(fuel_transactions)

[]
[]
[{'transaction_id': '10', 'client_id': '1559552960', 'transaction_sum_roubles': '326636.9294', 'purpose': 'Fuel', 'card_type': 'debit', 'transaction_datetime': '2026-06-04 01:20:50'}, {'transaction_id': '12', 'client_id': '3694885437', 'transaction_sum_roubles': '186316.0524', 'purpose': 'Fuel', 'card_type': 'debit', 'transaction_datetime': '2026-06-26 21:02:39'}, {'transaction_id': '27', 'client_id': '3283495036', 'transaction_sum_roubles': '329718.9048', 'purpose': 'Fuel', 'card_type': 'debit', 'transaction_datetime': '2026-06-30 01:02:57'}, {'transaction_id': '33', 'client_id': '6331965617', 'transaction_sum_roubles': '55390.1821', 'purpose': 'Fuel', 'card_type': 'debit', 'transaction_datetime': '2026-06-30 09:56:19'}, {'transaction_id': '43', 'client_id': '355414369', 'transaction_sum_roubles': '165523.7755', 'purpose': 'Fuel', 'card_type': 'debit', 'transaction_datetime': '2026-06-15 00:46:33'}, {'transaction_id': '45', 'client_id': '6147134662', 'transaction_sum_roubles': '

# 5

In [10]:
from functools import wraps
from typing import Callable

def log_func_call(
    func: Callable | None = None, *,
    show_args: bool = True, show_result: bool = True
) -> Callable:
    
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            if show_args:
                arguments = [repr(arg) for arg in args]
                arguments.extend(f"{key}={value!r}" for key, value in kwargs.items())
                call = f"{func.__name__}({', '.join(arguments)})"
            else:
                call = f"{func.__name__}(...)"
            print(f"Вызов: {call}")

            try:
                result = func(*args, **kwargs)
            except Exception as e:
                print(f"Ошибка: {call} -> {type(e).__name__}: {e}")
                raise
            
            if show_result:
                print(f"Результат: {call} -> {result!r}")
            return result
        
        return wrapper
    if func is None:
        return decorator
    return decorator(func)

In [11]:
@log_func_call
def add(a, b):
    return a + b


@log_func_call
def divide(a, b):
    return a / b

In [12]:
add(2, 3)
add(2, b=3)
print(add.__name__)  # add

Вызов: add(2, 3)
Результат: add(2, 3) -> 5
Вызов: add(2, b=3)
Результат: add(2, b=3) -> 5
add


In [13]:
try:
    divide(10, 0)
except ZeroDivisionError:
    pass

Вызов: divide(10, 0)
Ошибка: divide(10, 0) -> ZeroDivisionError: division by zero


In [14]:
@log_func_call(show_args=False, show_result=True)
def multiply(a, b):
    return a * b


@log_func_call(show_args=True, show_result=False)
def subtract(a, b):
    return a - b


result = multiply(2, 3)
result = subtract(10, 4)

Вызов: multiply(...)
Результат: multiply(...) -> 6
Вызов: subtract(10, 4)


# 6

LinearSpaceElement — абстрактный класс, который задаёт операции над элементами линейного пространства: сложение, вычитание, умножение и деление на число. Vector2D наследуется от LinearSpaceElement и реализует эти операции для двумерного вектора

In [15]:
from abc import ABC, abstractmethod
from math import sqrt


class LinearSpaceElement(ABC):
    @abstractmethod
    def __add__(self, other: "LinearSpaceElement") -> "LinearSpaceElement":
        pass

    @abstractmethod
    def __sub__(self, other: "LinearSpaceElement") -> "LinearSpaceElement":
        pass

    @abstractmethod
    def __mul__(self, other: int | float) -> "LinearSpaceElement":
        pass

    @abstractmethod
    def __truediv__(self, other: int | float) -> "LinearSpaceElement":
        pass


class Vector2D(LinearSpaceElement):
    def __init__(self, x: int | float, y: int | float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Vector2D({self.x!r}, {self.y!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Vector2D):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __add__(self, other: "Vector2D") -> "Vector2D":
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector2D") -> "Vector2D":
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, other: "Vector2D | int | float") -> "Vector2D | int | float":
        if isinstance(other, Vector2D):
            return self.x * other.x + self.y * other.y
        if isinstance(other, (int, float)):
            return Vector2D(self.x * other, self.y * other)
        return NotImplemented

    def __rmul__(self, other: int | float) -> "Vector2D":
        return self * other

    def __truediv__(self, other: int | float) -> "Vector2D":
        if not isinstance(other, (int, float)):
            return NotImplemented
        if other == 0:
            raise ZeroDivisionError("Нельзя делить вектор на ноль")
        return Vector2D(self.x / other, self.y / other)

    def length(self) -> float:
        return sqrt(self.x ** 2 + self.y ** 2)

In [16]:
a = Vector2D(3, 4)
b = Vector2D(1, 2)

print(a + b)       # Vector2D(4, 6)
print(a - b)       # Vector2D(2, 2)

Vector2D(4, 6)
Vector2D(2, 2)


# 7

In [17]:
import random
import string

def _make_user(user_id: int) -> dict:
    return {
        "id": user_id,
        "name": "".join(random.choices(string.ascii_uppercase, k=5)),
        "age": random.randint(18, 60),
        "email": "".join(random.choices(string.ascii_lowercase, k=6)) + "@gmail.com",
    }


def fetch_users(user_id: int | None = None) -> list[dict]:
    if user_id is None:
        ids = range(1, random.randint(2, 6))
    elif random.random() < 0.2:
        return []
    else:
        ids = [user_id]
    return [_make_user(i) for i in ids]

In [18]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class User:
    id: int
    name: str
    age: int
    email: str


def user_factory(api_func: Callable[[int | None], list[dict]]) -> Callable:
    def get_users(user_id: int | None = None) -> list[User] | User:
        users = [User(**user) for user in api_func(user_id)]
        if user_id is not None and users:
            return users[0]
        return users
    return get_users

In [19]:
get_users = user_factory(fetch_users)

all_users = get_users()
one_user = get_users(101)
nobody = get_users(99)

print(all_users)  # [User(...), User(...), ...]
print(one_user)   # User(id=101, ...) или []
print(nobody)     # []

[User(id=1, name='TSBDG', age=32, email='mdbfzq@gmail.com'), User(id=2, name='ZRRPV', age=43, email='uadhfd@gmail.com'), User(id=3, name='OGUGV', age=52, email='mcipsi@gmail.com'), User(id=4, name='SCSQI', age=37, email='qgdmgh@gmail.com'), User(id=5, name='ZLVAD', age=39, email='omfjau@gmail.com')]
[]
User(id=99, name='KXIFD', age=25, email='yddekh@gmail.com')


In [20]:
get_users_over_20 = user_factory(lambda user_id = None: [user for user in fetch_users(user_id) if user["age"] > 20])
users_over_20 = get_users_over_20()

print(users_over_20)
print(all(user.age > 20 for user in users_over_20))  # True

[User(id=1, name='HBAGU', age=46, email='ysqqoo@gmail.com'), User(id=2, name='VQUJK', age=31, email='phwxen@gmail.com'), User(id=3, name='WQPNJ', age=52, email='ffawpf@gmail.com'), User(id=4, name='ROEXM', age=34, email='syepet@gmail.com')]
True


# 8

In [21]:
class InsufficientFundsError(Exception):
    """Ошибка при попытке снять больше денег, чем есть на счёте."""


class BankAccount:
    def __init__(self, owner: str):
        self.owner = owner
        self._balance: float = 0

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")
        if amount > self._balance:
            raise InsufficientFundsError("Недостаточно средств")
        self._balance -= amount

    def __iadd__(self, amount: float) -> "BankAccount":
        self.deposit(amount)
        return self

    def __isub__(self, amount: float) -> "BankAccount":
        self.withdraw(amount)
        return self

    def __str__(self) -> str:
        return f"Счёт {self.owner}: {self._balance:g} руб."

    def __bool__(self) -> bool:
        return self._balance > 0

In [22]:
acc = BankAccount("Иван")

print(bool(acc))  # False

acc.deposit(1000)
acc += 500
acc -= 200

print(acc)          # Счёт Иван: 1300 руб.
print(acc.balance)  # 1300
print(bool(acc))    # True

try:
    acc.withdraw(5000)
except InsufficientFundsError:
    print("Недостаточно средств")

print(acc.balance)  # 1300

False
Счёт Иван: 1300 руб.
1300
True
Недостаточно средств
1300
